# Titanic 
## Score: .77033

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists(), "Set cwd to project root (folder containing titanic/train.csv)"

In [2]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title)
    rare = {
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    }
    out["Title"] = out["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )
    out.loc[out["Title"].isin(rare), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    return out

In [3]:
def build_pipeline() -> Pipeline:
    numeric = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "HasCabin"]
    categorical = ["Sex", "Embarked", "Title"]

    pre = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), numeric),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical,
            ),
        ]
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("prep", pre), ("model", clf)])

In [4]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

train_x = add_features(train.drop(columns=["Survived"]))
train_y = train["Survived"]
test_x = add_features(test)

pipe = build_pipeline()
pipe.fit(train_x, train_y)

pred = pipe.predict(test_x)
sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
out_path = ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(sub)} rows)")
sub.head(10)

Wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv (418 rows)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
